# P4: does the grammar help a model that can reason? (Qwen3-32B)

The 8B/14B spike showed the grammar guarantees valid queries (Neo4j errors 52% -> 0%, no invented names) but not accuracy: those models barely explore (0–2 queries per question). This notebook repeats free vs grammar with **Qwen3-32B-AWQ**, thinking **off** and **on**, on the clinic **train** questions (16 Q), both arms, judged by GLM-4.7 on OpenRouter.

| run | thinking | grammar |
|---|---|---|
| `p4_qwen3-32b_nothink_none` | off | free |
| `p4_qwen3-32b_nothink_cfg-xgrammar` | off | grammar |
| `p4_qwen3-32b_think_none` | on | free |
| `p4_qwen3-32b_think_cfg-xgrammar` | on | grammar |

1. **Runtime:** GPU, **A100** (Colab Pro). The 4-bit 32B model is ~19 GB; an L4 (24 GB) leaves too little room for a 32k context.
2. **Colab Secrets:** `OPENROUTER_API_KEY` with *Notebook access* on. `HF_TOKEN` optional.
3. **Runtime → Run all.** Approve the Drive prompt with the account that holds `MyDrive/chatdt` (andyoulisten).

Expect ~15 min setup (vLLM install + 19 GB download), ~10 min per thinking-off run, ~20–40 min per thinking-on run. Judge < $0.50 in total. Every run command is issued twice: the second pass only fills cells a transient error dropped (finished cells are skipped).

In [ ]:
# 1. Code + secrets
import os, subprocess
from google.colab import userdata

BRANCH = 'journal/agentic-cfg'
REPO = 'github.com/IvansSmirnoff/Chat.DT.git'

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/Chat.DT'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/Chat.DT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Chat.DT', 'pull', '-q', '--ff-only'], check=True)
os.chdir('/content/Chat.DT')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY') or ''
assert os.environ['OPENROUTER_API_KEY'], 'Add OPENROUTER_API_KEY to Colab Secrets'
os.environ.update({
    'NEO4J_URI': 'bolt://127.0.0.1:7687', 'NEO4J_USER': 'neo4j', 'NEO4J_PASSWORD': 'password',
    'IFCBENCH_MANIFEST_PATH': '/content/Chat.DT/data/ifcbench/manifest.yaml',
})

In [ ]:
# 2. GPU + dependencies + Neo4j. vLLM goes in after the base requirements, so its own pins win.
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU: Runtime -> Change runtime type -> L4 (or T4)'
driver_major = int(gpu.split(',')[1].strip().split('.')[0])
print('GPU:', gpu)

!pip install -q -r requirements-base.txt uv
VLLM = '0.32.0'
if driver_major >= 580:   # CUDA 13 builds need an R580+ driver
    !uv pip install --system -q vllm=={VLLM} --torch-backend=auto
else:                     # older driver: the CUDA 12.9 build from the release page
    !uv pip install --system -q "https://github.com/vllm-project/vllm/releases/download/v{VLLM}/vllm-{VLLM}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129 --index-strategy unsafe-best-match
# Colab's preinstalled torchaudio is built for Colab's torch; vLLM brings a newer torch, and transformers
# imports any torchaudio it finds, which then refuses the CUDA mismatch. Nothing here needs audio.
!pip uninstall -y -q torchaudio
!vllm download-kernels || echo '(kernel download skipped)'
subprocess.run(['vllm', '--version'], check=True)   # same imports as `vllm serve`: fail here, not in the background
!python -c "import torch; print('torch', torch.__version__, '| cuda ok:', torch.cuda.is_available())"

import psutil
ram_gb = psutil.virtual_memory().total / 2**30
os.environ['NEO4J_HEAP_MAX'] = '16g' if ram_gb > 60 else '4g'
os.environ['NEO4J_PAGECACHE'] = '8g' if ram_gb > 60 else '2g'
!bash scripts/colab/bootstrap.sh

In [ ]:
# 3. Persist snapshots + results on Drive (resume across sessions)
from google.colab import drive
drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/chatdt'
for local, remote in [('data/ifcbench/cache', 'ifcbench/cache'),
                      ('results/ifcbench/runs', 'ifcbench/runs')]:
    os.makedirs(f'{PERSIST}/{remote}', exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        subprocess.run(['rm', '-rf', local])
        os.symlink(f'{PERSIST}/{remote}', local)
    print(local, '->', os.readlink(local))

In [ ]:
# 4. IFC files from HuggingFace (pinned dataset commit, sha256-verified).
#    --project dental_clinic for Tier A; --split test for all 31 test-split files (1.34 GB).
!python scripts/ifcbench/fetch.py --project dental_clinic

In [ ]:
# 5. vLLM server helper. Backend xgrammar (llguidance runs out of its mask budget on this grammar).
#    Thinking-off runs: no reasoning parser, so the grammar covers the whole reply.
#    Thinking-on runs: --reasoning-parser qwen3, so the model thinks freely and the grammar starts after </think>.
import time, urllib.request
MODEL = 'Qwen/Qwen3-32B-AWQ'

def start_vllm(model, extra=()):
    subprocess.run(['pkill', '-f', 'vllm serve'], check=False)
    time.sleep(10)
    log = open('/content/vllm.log', 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', model, '--dtype', 'half', '--max-model-len', '32768',
         '--gpu-memory-utilization', '0.90', '--max-num-seqs', '8', '--port', '8000',
         '--structured-outputs-config.backend', 'xgrammar', *extra],
        stdout=log, stderr=subprocess.STDOUT)
    for i in range(360):                       # first start downloads ~19 GB: allow 30 min
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
            print(f'{model} up after {i * 5} s')
            return proc
        except Exception:
            time.sleep(5)
    print(open('/content/vllm.log').read()[-4000:])
    raise RuntimeError('vLLM did not come up; log tail above')

os.environ['VLLM_BASE_URL'] = 'http://127.0.0.1:8000/v1'

def run_pair(reasoning, tag):
    common = (f'--model-id dental_clinic__arc --model-id dental_clinic__str --split train '
              f'--llm vllm:{MODEL} --reasoning {reasoning} --judge-provider z-ai --workers 8')
    for grammar, suffix in (('none', 'none'), ('cfg', 'cfg-xgrammar')):
        run = f'p4_qwen3-32b_{tag}_{suffix}'
        for _ in range(2):                     # 2nd pass fills any cell a transient error dropped
            get_ipython().system(f'python -m src.agent.run {common} --grammar {grammar} --run {run}')   # streams like !

In [ ]:
# 6. Thinking off: serve, check the grammar is enforced, run free + grammar.
vllm_proc = start_vllm(MODEL)
import json, httpx
r = httpx.post('http://127.0.0.1:8000/v1/chat/completions', timeout=300, json={
    'model': MODEL, 'temperature': 0.1, 'max_tokens': 20, 'chat_template_kwargs': {'enable_thinking': False},
    'messages': [{'role': 'user', 'content': 'Write a short poem about doors.'}],
    'structured_outputs': {'grammar': 'start: "yes" | "no"'}})
reply = r.json()['choices'][0]['message']['content']
print('constrained reply:', repr(reply))
assert reply.strip() in ('yes', 'no'), 'grammar NOT applied: stop here'
run_pair('off', 'nothink')

In [ ]:
# 7. Thinking on: restart with the qwen3 reasoning parser, run free + grammar.
vllm_proc = start_vllm(MODEL, extra=('--reasoning-parser', 'qwen3'))
run_pair('on', 'think')

In [ ]:
# 8. Compare every P4 run found on Drive (8B/14B from the spike, 32B from here).
import json, pandas as pd
rows = []
for run in sorted(os.listdir('results/ifcbench/runs')):
    path = f'results/ifcbench/runs/{run}/summary.json'
    if not run.startswith('p4_') or not os.path.exists(path) or run.endswith('_cfg'):   # *_cfg = cut-off llguidance runs
        continue
    for paradigm, c in json.load(open(path)).items():
        rows.append({'run': run, 'arm': paradigm, 'correct': f"{c['correct']}/{c['n']}",
                     'abstained': c['abstained'], 'queries/q': c['mean_queries'],
                     'query errors': c.get('query_error_rate'), 'token limit': c['token_limit'],
                     'parse errors': c['parse_error'], 's/q': c['mean_wall_s']})
display(pd.DataFrame(rows))

# Share of the free runs' error-free queries naming something the graph lacks (what the grammar forbids).
for run in ('p4_qwen3-32b_nothink_none', 'p4_qwen3-32b_think_none'):
    print(run)
    get_ipython().system(f'python scripts/grammar_coverage.py results/ifcbench/runs/{run}/episodes.jsonl '
                         f'--manifest data/ifcbench/manifest.yaml | head -3')